# Experiment 1.1 — Intrinsic Dataset Quality Evaluation

## Research Question
How can we assess the intrinsic quality of an automatically generated concept dataset?

This notebook uses a **VLM-as-judge** to measure how well each image actually depicts its concept. The judge is `Qwen2-VL-2B-Instruct`.

1. **Prompt:** The judge is asked whether `{concept}` is the *dominant* visible pattern covering most of the image *with no other competing pattern*, and is told to answer "yes" only if both conditions hold.

2. **Logit-based scoring:** Rather than letting the VLM generate a free-form yes/no answer, we run a single forward pass and read the next-token logits at the position the assistant would begin its reply. We softmax those logits over the candidate tokens for "Yes" / "No" (all casing variants combined via `logsumexp`) to obtain a continuous **P(yes | image, concept)** score per image. This is the standard approach in G-Eval, Prometheus, and the broader LLM-as-judge literature; it removes generation sycophancy and gives back a calibrated alignment score.

CLIP is still used as a frozen feature extractor for two embedding-based dataset-quality metrics (repetition rate, participation ratio) — these answer questions about the *embedding space* of the dataset (duplication, spread), not about whether each image matches the concept.

### Metrics computed per concept (Baseline vs. Automatic dataset):
| Metric | Description |
|--------|-------------|
| **VLM Alignment — Mean** | Mean P(yes) across images | 
| **VLM Alignment — Std** | Spread of P(yes) |
| **VLM Alignment — % below 0.25** | Fraction of clearly off-concept images |
| **VLM Precision @ 0.5** | % of images with P(yes) ≥ 0.5 |
| **VLM Precision @ 0.7** | % of images with P(yes) ≥ 0.7 (strict) |
| **Repetition Rate** | % of near-duplicate pairs (CLIP cosine sim > τ=0.95) |
| **Participation Ratio** | Effective rank of CLIP embedding covariance (diversity proxy) |

In [ ]:
!pip install --quiet open_clip_torch pandas "transformers>=4.45" accelerate

In [ ]:
import os
import json
import shutil
import tempfile
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import open_clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# CLIP (used here ONLY as a frozen feature extractor for repetition rate and participation ratio — no concept scoring)
model_name  = "ViT-B-32"
pretrained  = os.environ.get("CLIP_PRETRAINED", "openai")

hf_token = os.environ.get("HF_TOKEN", None)
if hf_token:
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", hf_token)
elif pretrained != "openai":
    print(f"WARNING: pretrained='{pretrained}' may require an HF token. Falling back to 'openai'.")
    pretrained = "openai"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    model_name, pretrained=pretrained
)
clip_model = clip_model.to(device).eval()
print(f"CLIP loaded (embeddings only): {model_name} / {pretrained}")

In [ ]:
# VLM-as-Judge: load Qwen2-VL-2B-Instruct
# Other choices could be: Qwen/Qwen2.5-VL-3B-Instruct, Qwen/Qwen2-VL-7B-Instruct
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

VLM_MODEL_ID = os.environ.get("VLM_MODEL_ID", "Qwen/Qwen2-VL-2B-Instruct")
vlm_dtype    = torch.float16 if device == "cuda" else torch.float32

print(f"Loading VLM judge: {VLM_MODEL_ID}  (first run downloads ~5 GB) ...")
vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=vlm_dtype,
    device_map="auto",
).eval()
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
print("VLM judge loaded.")

# Discover token IDs for "Yes" and "No" (all casing variants)
# Used by the logit-based judge to read P(yes) from the model's first-token logits.
def _discover_yn_token_ids(processor):
    tok = processor.tokenizer
    yes_ids, no_ids = [], []
    for v in ["Yes", " Yes", "yes", " yes", "YES"]:
        ids = tok.encode(v, add_special_tokens=False)
        if len(ids) == 1:
            yes_ids.append(ids[0])
    for v in ["No", " No", "no", " no", "NO"]:
        ids = tok.encode(v, add_special_tokens=False)
        if len(ids) == 1:
            no_ids.append(ids[0])
    return sorted(set(yes_ids)), sorted(set(no_ids))

YES_TOKEN_IDS, NO_TOKEN_IDS = _discover_yn_token_ids(vlm_processor)
print(f"Yes token IDs: {YES_TOKEN_IDS}")
print(f"No token IDs:  {NO_TOKEN_IDS}")
assert YES_TOKEN_IDS and NO_TOKEN_IDS, "Failed to find single-token Yes/No variants — judge will not work."

## Section 3 — Dataset Paths Configuration

In [ ]:
ACG_NEW = "/kaggle/input/datasets/alessandrocogollo/acg4cav-base-extraction/dataset"

CROP_METHODS = ["bbox", "center_mask", "largest_bbox", "sliding_window"]
SPLITS       = ["single", "multi", "random"]

DTD2_ROOT = "/kaggle/input/datasets/jmexpert/describable-textures-dataset-dtd/dtd/images"
T2I_ROOT  = "/kaggle/input/datasets/alessandrocogollo/zeroshot-t2i-concepts/concepts"

MANUAL_BASELINE: Dict[str, str] = {
    "striped":   f"{DTD2_ROOT}/striped",
    "dotted":    f"{DTD2_ROOT}/dotted",
    "chequered": f"{DTD2_ROOT}/chequered",
    "wood":      f"{T2I_ROOT}/wood/wood",
    "water":     f"{T2I_ROOT}/water/water",
    "braided":   f"{DTD2_ROOT}/braided",
    "bubbly":    f"{T2I_ROOT}/bubbly/bubbly",
    "fibrous":   f"{DTD2_ROOT}/fibrous",
    "veined":    f"{DTD2_ROOT}/veined",
}

CONCEPTS_MAIN  = ["striped", "dotted", "chequered", "wood", "water", "braided", "bubbly", "fibrous", "veined"]
ALL_CONCEPTS   = CONCEPTS_MAIN # mappatura forzata

# Builder mapping
def build_mapping_auto_vanilla(concepts: List[str], crop_method: str, split: str) -> Dict[str, Dict[str, str]]:
    aug_root = f"{ACG_NEW}/augmented/augmented_{crop_method}"
    van_root = f"{ACG_NEW}/vanilla/vanilla_{crop_method}"
    return {
        c: {
            "augmented":    f"{aug_root}/{c}/{split}",
            "vanilla": f"{van_root}/{c}/{split}",
        }
        for c in concepts
    }

def build_mapping_manual(concepts: List[str]) -> Dict[str, Dict[str, str]]:
    return {c: {"manual": MANUAL_BASELINE[c]} for c in concepts}


THRESHOLD_REPETITION = 0.95
BATCH_SIZE           = 64
MAX_IMAGES_PER_DIR   = None
THRESHOLD_ALIGNMENT  = 0.25

print(f"Config OK — {len(CROP_METHODS)} crop methods × {len(SPLITS)} splits × {len(ALL_CONCEPTS)} concepts")
print(f"Total (crop, split) cells: {len(CROP_METHODS) * len(SPLITS)}")

## Section 4 — Core Utilities (CLIP embeddings + VLM judge)


In [ ]:
# Image helpers
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def list_images(img_dir: str, max_n: int = MAX_IMAGES_PER_DIR) -> List[str]:
    """Return sorted list of image file paths inside img_dir, optionally capped at max_n."""
    paths = sorted(
        os.path.join(img_dir, f)
        for f in os.listdir(img_dir)
        if os.path.splitext(f.lower())[1] in IMAGE_EXTS
    )
    if max_n is not None:
        paths = paths[:max_n]
    return paths

# CLIP embeddings (for repetition rate + participation ratio only)

@torch.no_grad()
def embed_images(img_paths: List[str], batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, List[str]]:
    """
    Batch-embed images with CLIP.
    Skips unreadable files silently.
    Returns (N, D) L2-normalised float32 array and the valid path list.
    """
    embs, valid = [], []
    for i in range(0, len(img_paths), batch_size):
        batch_t, batch_p = [], []
        for p in img_paths[i : i + batch_size]:
            try:
                batch_t.append(clip_preprocess(Image.open(p).convert("RGB")))
                batch_p.append(p)
            except Exception:
                continue
        if not batch_t:
            continue
        feats = clip_model.encode_image(torch.stack(batch_t).to(device))
        feats = feats / feats.norm(dim=-1, keepdim=True)
        embs.append(feats.cpu().float().numpy())
        valid.extend(batch_p)
    if not embs:
        return np.zeros((0, clip_model.visual.output_dim), dtype=np.float32), []
    return np.vstack(embs).astype(np.float32), valid


def repetition_rate_from_embs(embs: np.ndarray, tau: float = THRESHOLD_REPETITION) -> dict:
    """Near-duplicate detection on pre-computed L2-normalised CLIP embeddings."""
    N = embs.shape[0]
    if N < 2:
        return {"N": N, "tau": tau, "repetition_rate": 0.0, "num_pairs": 0,
                "num_high_sim_pairs": 0, "max_offdiag_sim": None, "mean_offdiag_sim": None}
    S    = embs @ embs.T
    iu   = np.triu_indices(N, k=1)
    sims = S[iu]
    n_high = int((sims > tau).sum())
    return {
        "N": N,
        "tau": tau,
        "repetition_rate":   n_high / sims.size,
        "num_pairs":         int(sims.size),
        "num_high_sim_pairs": n_high,
        "max_offdiag_sim":   float(sims.max()),
        "mean_offdiag_sim":  float(sims.mean()),
    }


def participation_ratio_from_embs(embs: np.ndarray) -> dict:
    """
    Participation Ratio (PR) = (Σλ_i)² / Σ(λ_i²)  where λ_i are eigenvalues of the
    embedding covariance matrix.
    High PR → embeddings span many directions → diverse dataset.
    Low PR  → embeddings collapse to a few modes.
    """
    N = embs.shape[0]
    if N < 2:
        return {"N": N, "participation_ratio": float("nan")}
    Z_c     = embs - embs.mean(axis=0, keepdims=True)
    cov     = (Z_c.T @ Z_c) / (N - 1)
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = eigvals[eigvals > 0]
    pr = float(eigvals.sum() ** 2 / (eigvals ** 2).sum())
    return {"N": N, "participation_ratio": pr}


# VLM-as-Judge (logit-based, strict prompt)

JUDGE_PROMPT_TMPL = (
    "Look at this image. Is '{concept}' the dominant visible pattern?\n"
    "Answer 'yes' only if the conditions hold; otherwise answer 'no'."
)

THRESHOLD_ALIGNMENT = 0.25  # P(yes) below this is considered "clearly off-concept"


@torch.inference_mode()
def vlm_judge_prob(img_path: str, concept: str) -> float:
    """
    Return P(yes | image, concept) from a single VLM forward pass.

    We read the next-token logits at the position the assistant begins its reply,
    select the logits at the candidate "Yes" and "No" token IDs (all casing
    variants), combine multi-variant logits with logsumexp, and softmax over
    the two combined scalars. This avoids sycophancy from generative decoding.

    Returns NaN for unreadable images.
    """
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception:
        return float("nan")

    prompt_text = JUDGE_PROMPT_TMPL.format(concept=concept)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": prompt_text},
        ],
    }]
    text   = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text], images=[img], padding=True, return_tensors="pt",
    ).to(vlm_model.device)

    out         = vlm_model(**inputs)
    last_logits = out.logits[0, -1, :]                             # (vocab,)
    yes_lse     = torch.logsumexp(last_logits[YES_TOKEN_IDS], dim=0)
    no_lse      = torch.logsumexp(last_logits[NO_TOKEN_IDS],  dim=0)
    p_yes       = torch.softmax(torch.stack([yes_lse, no_lse]), dim=0)[0]
    return float(p_yes.item())


def vlm_judge_probs_batch(img_paths: List[str], concept: str) -> List[float]:
    """Return P(yes) per image. Sequential (one forward pass per image)."""
    try:
        from tqdm.auto import tqdm
    except ImportError:
        tqdm = lambda x, **k: x
    return [vlm_judge_prob(p, concept) for p in tqdm(img_paths, desc=f"VLM '{concept}'", leave=False)]


def compute_vlm_metrics(
    probs: List[float],
    threshold_alignment: float = THRESHOLD_ALIGNMENT,
    threshold_lenient: float = 0.5,
    threshold_strict: float  = 0.7,
) -> dict:
    """Aggregate per-image P(yes) into the alignment + precision metrics."""
    nan = float("nan")
    arr = np.array(probs, dtype=np.float32)
    arr = arr[~np.isnan(arr)]   # drop unreadable images
    n   = int(arr.size)
    if n == 0:
        return {
            "N": 0,
            "vlm_align_mean":      nan,
            "vlm_align_std":       nan,
            "vlm_align_pct_below": nan,
            "vlm_prec_05":         nan,
            "vlm_prec_07":         nan,
        }
    return {
        "N":                   n,
        "vlm_align_mean":      float(arr.mean()),
        "vlm_align_std":       float(arr.std(ddof=1)) if n > 1 else nan,
        "vlm_align_pct_below": float((arr < threshold_alignment).mean() * 100),
        "vlm_prec_05":         float((arr >= threshold_lenient).mean() * 100),
        "vlm_prec_07":         float((arr >= threshold_strict).mean()  * 100),
    }


## Section 5 — Main Experiment Loop

Results are **checkpointed** to `RESULTS_PATH` after each concept,
so a crashed or interrupted run can be resumed without re-embedding everything.


In [ ]:
# Run all metrics for every concept
def experiments_run(MAPPINGS, RESULTS_PATH, tag, force=False):
    all_results = {}

    # Resume from checkpoint if available
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            all_results = json.load(f)
        print(f"Resumed from checkpoint — {len(all_results)} concept(s) already done.")

    for concept, paths_dict in MAPPINGS.items():
        key = f"{tag}:{concept}"
    
        if not force and key in all_results:
            print(f"[SKIP] {concept}  (checkpoint)")
            continue
    
        print(f"\n{'='*60}\n  Concept: '{concept}'\n{'='*60}")
        concept_result = {}
    
        for split_name, path in paths_dict.items():
            if not path or not os.path.isdir(path):
                print(f"  [{split_name}] Path missing — skipping.")
                concept_result[split_name] = None
                continue

            paths  = list_images(path)
            n_imgs = len(paths)
            print(f"  [{split_name}] {n_imgs} images  |  {path}")

            print(f"  [{split_name}] Embedding images (CLIP, for diversity metrics) ...")
            embs, paths_valid = embed_images(paths)

            print(f"  [{split_name}] VLM-as-judge logit scoring ({len(paths_valid)} images) ...")
            judge_probs   = vlm_judge_probs_batch(paths_valid, concept)
            judge_metrics = compute_vlm_metrics(judge_probs)

            print(f"  [{split_name}] Repetition rate ...")
            rep = repetition_rate_from_embs(embs)

            print(f"  [{split_name}] Participation ratio (diversity) ...")
            div = participation_ratio_from_embs(embs)

            concept_result[split_name] = {
                "judge_probs":   judge_probs,
                "judge_metrics": judge_metrics,
                "rep":           rep,
                "div":           div,
            }
            print(f"  [{split_name}] Align μ={judge_metrics['vlm_align_mean']:.3f}  "
                  f"σ={judge_metrics['vlm_align_std']:.3f}  "
                  f"%<0.25: {judge_metrics['vlm_align_pct_below']:.1f}%  "
                  f"P@0.5: {judge_metrics['vlm_prec_05']:.1f}%  "
                  f"P@0.7: {judge_metrics['vlm_prec_07']:.1f}%  "
                  f"Rep: {rep['repetition_rate']*100:.2f}%  "
                  f"PR: {div['participation_ratio']:.2f}")

        all_results[key] = concept_result

        with tempfile.NamedTemporaryFile("w", delete=False, suffix=".json") as tmp:
            json.dump(all_results, tmp)
        shutil.move(tmp.name, RESULTS_PATH)

    print("\n✓ All concepts processed.")
    return all_results

In [ ]:
manual_results = experiments_run(
    build_mapping_manual(ALL_CONCEPTS),
    "results_manual.json",
    "manual",
)

auto_vanilla_results: Dict[str, dict] = {}   # tag → results dict

for crop_method in CROP_METHODS:
    for split in SPLITS:
        tag = f"{crop_method}:{split}"
        print(f"\n{'#'*70}\n#  {tag}\n{'#'*70}")
        auto_vanilla_results[tag] = experiments_run(
            build_mapping_auto_vanilla(CONCEPTS_MAIN, crop_method, split),
            f"results_{crop_method}_{split}.json",
            tag,
        )

for crop_method in CROP_METHODS:
    tag = f"{crop_method}:orthogonal"
    print(f"\n{'#'*70}\n#  {tag}\n{'#'*70}")
    auto_vanilla_results[tag] = experiments_run(
        build_mapping_auto_vanilla(CONCEPTS_ORTHO, crop_method, "orthogonal"),
        f"results_{crop_method}_orthogonal.json",
        tag,
    )

## Section 6 — Results
An aggregate table (and then CSV) with the results

In [ ]:
import pandas as pd
import numpy as np

try:
    from IPython.display import display as ipy_display
    _display = ipy_display
except ImportError:
    _display = print

THRESHOLD_ALIGNMENT = 0.25
METRIC_INFO = [
    ("VLM Alignment — Mean",                          "vlm_align_mean",      "↑"),
    ("VLM Alignment — Std",                           "vlm_align_std",       "↓"),
    (f"VLM Alignment — % below {THRESHOLD_ALIGNMENT}", "vlm_align_pct_below", "↓"),
    ("VLM Precision @ 0.5 (%)",                       "vlm_prec_05",         "↑"),
    ("VLM Precision @ 0.7 (%)",                       "vlm_prec_07",         "↑"),
    ("Repetition Rate (%)",                           "rep_rate",            "↓"),
    ("Num Near-Duplicate Pairs",                      "n_dup_pairs",         "↓"),
    ("Participation Ratio",                           "participation_ratio", "↑"),
    ("Dataset Size (N)",                              "N",                   "·"),
]

METRIC_INFO_MAP = {label: direction for label, _, direction in METRIC_INFO}

def _split_metrics(split_data) -> dict:
    nan = float("nan")
    if split_data is None or not isinstance(split_data, dict):
        return {k: nan for k in [
            "vlm_align_mean", "vlm_align_std", "vlm_align_pct_below",
            "vlm_prec_05", "vlm_prec_07",
            "rep_rate", "n_dup_pairs", "participation_ratio", "N"
        ]}
    judge = split_data.get("judge_metrics", {}) or {}
    rep   = split_data.get("rep", {"repetition_rate": nan, "num_high_sim_pairs": nan})
    div   = split_data.get("div", {"participation_ratio": nan, "N": nan})
    
    return {
        "vlm_align_mean":      judge.get("vlm_align_mean",      nan),
        "vlm_align_std":       judge.get("vlm_align_std",       nan),
        "vlm_align_pct_below": judge.get("vlm_align_pct_below", nan),
        "vlm_prec_05":         judge.get("vlm_prec_05",         nan),
        "vlm_prec_07":         judge.get("vlm_prec_07",         nan),
        "rep_rate":            rep["repetition_rate"] * 100 if not np.isnan(rep["repetition_rate"]) else nan,
        "n_dup_pairs":         rep["num_high_sim_pairs"],
        "participation_ratio": div["participation_ratio"],
        "N":                   div["N"],
    }

def _highlight_best_overall(df):
    def style_row(row):
        direction = METRIC_INFO_MAP.get(row.name, "·")
        styles = [""] * len(row)
        
        if direction in ["—", "·"]: 
            return styles
            
        cols_to_compare = [c for c in row.index if c != "Manual Baseline"]
        
        best_val = None
        for c in cols_to_compare:
            try:
                val = float(row[c])
                if not np.isnan(val):
                    if best_val is None:
                        best_val = val
                    else:
                        if direction == "↑":
                            best_val = max(best_val, val)
                        elif direction == "↓":
                            best_val = min(best_val, val)
            except:
                pass
                
        for i, c in enumerate(row.index):
            if c in cols_to_compare:
                try:
                    val = float(row[c])
                    if not np.isnan(val) and best_val is not None and np.isclose(val, best_val, atol=1e-5):
                        styles[i] = "background-color: #d6f5d6; font-weight: bold; color: #004d00;"
                except:
                    pass
        return styles
        
    return df.style.apply(style_row, axis=1)

all_concepts = set()
for strategy_results in auto_vanilla_results.values():
    for key in strategy_results.keys():
        all_concepts.add(key.split(":")[-1])
all_concepts = sorted(list(all_concepts))

print("  COMPREHENSIVE CONCEPT ANALYSIS (All Strategies vs Manual Baseline)")

for concept in all_concepts:
    cols_data = {}
    
    manual_cell = manual_results.get(f"manual:{concept}", {})
    cols_data["Manual Baseline"] = _split_metrics(manual_cell.get("manual"))
    
    for strategy_tag, strategy_results in auto_vanilla_results.items():
        cell_result = next((v for k, v in strategy_results.items() if k.endswith(f":{concept}") or k == concept), {})
        
        short_tag = strategy_tag.replace("augmented_", "").replace("vanilla_", "")
        
        cols_data[f"Van ({short_tag})"] = _split_metrics(cell_result.get("vanilla"))
        cols_data[f"Aug ({short_tag})"] = _split_metrics(cell_result.get("augmented"))

    df = pd.DataFrame(
        {col_name: [cols_data[col_name][key] for _, key, _ in METRIC_INFO] 
         for col_name in cols_data.keys()},
        index=[name for name, _, _ in METRIC_INFO]
    )
    df.index.name = "Metric"
    
    print(f"\n\n CONCEPT: '{concept.upper()}'")
    
    styled_df = _highlight_best_overall(df).format(precision=3, na_rep="NaN")
    _display(styled_df)